# Prompt Preview

Ce notebook lit `input/swissdox_2025_tagged.csv`, formate chaque article dans le template de prompt défini dans `src/run3_prompts.py` (avec mots-clés institutionnels), et produit un CSV avec :
- **`prompt_ready`** : le prompt complet à coller dans un chat LLM
- **`llm_answer`** : colonne vide à remplir manuellement avec la réponse du LLM

La colonne `text` brute est exclue pour alléger le fichier.

In [1]:
import sys
import pandas as pd
from pathlib import Path

# Ajouter src/ au path pour importer run3_prompts.py
sys.path.insert(0, str(Path('..').resolve() / 'src'))
from run3_prompts import build_user_prompt, SYSTEM_PROMPT

In [2]:
# Charger les données
df = pd.read_csv('../input/swissdox_2025_tagged.csv')
print(f"Articles chargés : {len(df):,}")
print(f"Colonnes : {df.columns.tolist()}")
df.head(2)

Articles chargés : 19,291
Colonnes : ['article_id', 'title', 'lead', 'text', 'pubtime', 'medium_code', 'medium_name', 'rubric', 'regional', 'doctype', 'doctype_description', 'language', 'char_count', 'dateline', 'matched_keywords']


,article_id,title,lead,text,pubtime,medium_code,medium_name,rubric,regional,doctype,doctype_description,language,char_count,dateline,matched_keywords
0,00027f05-690b-f341-5519-fcc50dbc5549,Martin Pfister envisage l’abandon du projet à ...,Le Conseiller fédéral envisage d’abandonner l’...,Le Conseiller fédéral envisage d’abandonner l’...,2025-07-05,ZWSO,20 minutes online,NaN,NaN,WWE,Online medium,fr,2813.0,NaN,Martin Pfister|Armée suisse
1,000658f2-9b7b-c9b4-9592-008f8f4d0b54,Vers la fin des vols en classe affaires pour l...,Privilèges Albert Rösti propose de réviser les...,Privilèges Albert Rösti propose de réviser les...,2025-09-30,HEU,24 heures,Actualités,NaN,PRD,Regional daily newspaper,fr,1612.0,NaN,Guy Parmelin|Albert Rösti


In [ ]:
# Répliquer le pipeline cluster : explode par keyword, puis construire les prompts
def explode_by_keyword(df):
    rows = df.copy()
    rows["keyword"] = rows["matched_keywords"].str.split("|")
    exploded = rows.explode("keyword").copy()
    exploded["keyword"] = exploded["keyword"].str.strip()
    exploded = exploded[exploded["keyword"].notna() & (exploded["keyword"] != "")].reset_index(drop=True)
    return exploded

df = explode_by_keyword(df)
print(f"Après explode : {len(df):,} lignes (article × keyword)")

# Construire la colonne prompt_ready
df['prompt_ready'] = df.apply(
    lambda row: SYSTEM_PROMPT + '\n\n' + build_user_prompt(row, 'text'), axis=1
)

df['llm_answer'] = ''

print(f"Exemple de prompt_ready pour le premier article :")
print("-" * 60)
print(df['prompt_ready'].iloc[0])

In [4]:
# Supprimer la colonne text brute et exporter
cols_to_keep = [c for c in df.columns if c != 'text']
df_out = df[cols_to_keep]

out_path = Path('../data/raw/swissdox_prompts_run3.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_path, index=False)

print(f"Fichier exporté : {out_path}")
print(f"Colonnes : {df_out.columns.tolist()}")
print(f"Lignes : {len(df_out):,}")

Fichier exporté : ../data/raw/swissdox_prompts_run3.csv
Colonnes : ['article_id', 'title', 'lead', 'pubtime', 'medium_code', 'medium_name', 'rubric', 'regional', 'doctype', 'doctype_description', 'language', 'char_count', 'dateline', 'matched_keywords', 'prompt_ready', 'llm_answer']
Lignes : 19,291


In [5]:
# Aperçu du résultat final
df_out[['article_id', 'medium_name', 'pubtime', 'matched_keywords', 'prompt_ready', 'llm_answer']].head(3)

,article_id,medium_name,pubtime,matched_keywords,prompt_ready,llm_answer
0,00027f05-690b-f341-5519-fcc50dbc5549,20 minutes online,2025-07-05,Martin Pfister|Armée suisse,\n\nYou will receive an article to analyze. Yo...,
1,000658f2-9b7b-c9b4-9592-008f8f4d0b54,24 heures,2025-09-30,Guy Parmelin|Albert Rösti,\n\nYou will receive an article to analyze. Yo...,
2,00068d8f-18c9-3631-a24f-362857e4ad9e,24heures.ch,2025-04-07,Guy Parmelin|Karin Keller-Sutter,\n\nYou will receive an article to analyze. Yo...,
